# IMPORTS:

In [16]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import optuna
from optuna.pruners import MedianPruner
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_selection import RFE, SelectKBest, f_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
df = pd.read_csv("cardio_data_processed.csv")

# Preprocessing & Feature Engineering:


In [3]:
df.drop('id', axis=1, inplace = True) # Dropped id becoz its useless for prediction...
df["age"] = (df.age/365).astype(int) # Converting Age in years as its given in days...
df.drop('age_years', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...
df.drop('bp_category_encoded', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...
df.drop('bp_category', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...

In [4]:
df["pulse_pressure"] = df["ap_hi"] - df["ap_lo"]
df["mean_arterial_pressure"] = df["ap_hi"] + 2*df["ap_lo"]/3

In [5]:
#Checking if there any missing or false value & dropping that...
df[['ap_hi', 'ap_lo', 'pulse_pressure', 'mean_arterial_pressure']].describe()
print((df['pulse_pressure'] < 0).sum()) #Since i got 3 value whose pulse_pressure is < 0 which make no sense as there is an error, so i gonna drop this 3 rows
df = df[df['pulse_pressure'] >= 0] #It removed those 3rows but index still show 0 to 68204 so we need to reset index...
df = df.reset_index(drop=True) #index reset...

3


# Train Test Split:

In [6]:
X = df.drop('cardio', axis = 1)
y = df['cardio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 10)

# OPTUNA for GNB:

In [9]:
def objective(trial):
    """
    Objective function for Optuna to optimize GNB hyperparameters
    """
    
    # Define hyperparameter search space for GNB
    var_smoothing = trial.suggest_float('var_smoothing', 1e-10, 1e-6, log=True)
    
    try:
        # Create pipeline with suggested hyperparameters
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", GaussianNB(
                var_smoothing=var_smoothing
            ))
        ])
        
        # Use Stratified 5-Fold CV to evaluate
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)
        
        # Define scoring metrics
        scoring = {
            'accuracy': 'accuracy',
            'precision': 'precision_weighted',
            'recall': 'recall_weighted',
            'f1': 'f1_weighted',
            'roc_auc': 'roc_auc_ovr_weighted'
        }
        
        # Perform cross-validation
        cv_results = cross_validate(
            pipeline, 
            X_train, 
            y_train, 
            cv=cv, 
            scoring=scoring,
            return_train_score=False,
            n_jobs=1
        )
        
        # Return mean Accuracy for optimization
        mean_accuracy = cv_results['test_accuracy'].mean()
        
        return mean_accuracy
        
    except Exception as e:
        # If any error occurs, return a very poor score
        print(f"Trial failed with error: {str(e)[:50]}")
        return 0.0


# Create study object
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=10, n_startup_trials=20),
    pruner=optuna.pruners.PercentilePruner(percentile=25)
)

# Run optimization
print("Starting Optuna optimization for Gaussian Naive Bayes...")
print("=" * 70)

study.optimize(
    objective, 
    n_trials=100, 
    show_progress_bar=True,
    gc_after_trial=True
)

# Get best trial
best_trial_gnb = study.best_trial

print("\n" + "=" * 70)
print("OPTUNA TUNING RESULTS - GAUSSIAN NAIVE BAYES")
print("=" * 70)
print(f"\nBest Accuracy: {best_trial_gnb.value:.4f}")
print(f"Number of Completed Trials: {len([t for t in study.trials if t.state.name == 'COMPLETE'])}")
print(f"\nBest Hyperparameters:")
print("-" * 70)
for key, value in best_trial_gnb.params.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.10f}")
    else:
        print(f"  {key}: {value}")

print("=" * 70)

# Store best params for next steps
best_params_gnb = best_trial_gnb.params

[I 2026-05-06 22:14:13,881] A new study created in memory with name: no-name-71d521bd-34a1-43ea-8017-3a529b013172


Starting Optuna optimization for Gaussian Naive Bayes...


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-05-06 22:14:14,821] Trial 0 finished with value: 0.7189749479554772 and parameters: {'var_smoothing': 1.2169775677700545e-07}. Best is trial 0 with value: 0.7189749479554772.
[I 2026-05-06 22:14:15,877] Trial 1 finished with value: 0.7189749479554772 and parameters: {'var_smoothing': 1.2106198691436024e-10}. Best is trial 0 with value: 0.7189749479554772.
[I 2026-05-06 22:14:16,685] Trial 2 finished with value: 0.7189749479554772 and parameters: {'var_smoothing': 3.4244666391252975e-08}. Best is trial 0 with value: 0.7189749479554772.
[I 2026-05-06 22:14:17,545] Trial 3 finished with value: 0.7189749479554772 and parameters: {'var_smoothing': 9.890438121030044e-08}. Best is trial 0 with value: 0.7189749479554772.
[I 2026-05-06 22:14:18,391] Trial 4 finished with value: 0.7189749479554772 and parameters: {'var_smoothing': 9.86343187233008e-09}. Best is trial 0 with value: 0.7189749479554772.
[I 2026-05-06 22:14:19,226] Trial 5 finished with value: 0.7189749479554772 and paramete

# RFE feature selection:

In [19]:
# Feature count range to test (8-14 features)
feature_counts = range(8, 15)  # 8, 9, 10, 11, 12, 13, 14

# Dictionary to store results
feature_selection_results_gnb = {}

# Stratified 5-Fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

# Loop through each feature count
for n_features in feature_counts:
    print(f"\nTesting with {n_features} features...")
    
    # Store metrics for all folds
    fold_accuracies = []
    fold_precisions = []
    fold_recalls = []
    fold_f1s = []
    fold_roc_aucs = []
    
    # CV Loop - FEATURE SELECTION EMBEDDED INSIDE
    fold_idx = 0
    for train_idx, val_idx in cv.split(X_train, y_train):
        fold_idx += 1
        
        # Split data into train and validation
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Step 1: Scale the training fold
        scaler = StandardScaler()
        X_fold_train_scaled = scaler.fit_transform(X_fold_train)
        X_fold_val_scaled = scaler.transform(X_fold_val)
        
        # Step 2: Select best features using SelectKBest (fit on training fold only)
        selector = SelectKBest(score_func=f_classif, k=n_features)
        X_fold_train_selected = selector.fit_transform(X_fold_train_scaled, y_fold_train)
        X_fold_val_selected = selector.transform(X_fold_val_scaled)
        
        # Step 3: Train GNB model on selected features
        model = GaussianNB(var_smoothing=best_params_gnb['var_smoothing'])
        model.fit(X_fold_train_selected, y_fold_train)
        
        # Step 4: Predict on validation fold
        y_pred = model.predict(X_fold_val_selected)
        y_pred_proba = model.predict_proba(X_fold_val_selected)[:, 1]
        
        # Calculate metrics
        accuracy = accuracy_score(y_fold_val, y_pred)
        precision = precision_score(y_fold_val, y_pred, average='weighted')
        recall = recall_score(y_fold_val, y_pred, average='weighted')
        f1 = f1_score(y_fold_val, y_pred, average='weighted')
        roc_auc = roc_auc_score(y_fold_val, y_pred_proba, average='weighted')
        
        # Store results
        fold_accuracies.append(accuracy)
        fold_precisions.append(precision)
        fold_recalls.append(recall)
        fold_f1s.append(f1)
        fold_roc_aucs.append(roc_auc)
    
    # Store mean and std for this feature count
    feature_selection_results_gnb[n_features] = {
        'accuracy': np.array(fold_accuracies),
        'precision': np.array(fold_precisions),
        'recall': np.array(fold_recalls),
        'f1': np.array(fold_f1s),
        'roc_auc': np.array(fold_roc_aucs)
    }
    
    # Display results for this feature count
    mean_accuracy = np.mean(fold_accuracies)
    std_accuracy = np.std(fold_accuracies)
    print(f"  Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")

# ============================================================
# SELECT BEST FEATURE COUNT (based on Accuracy)
# ============================================================

print("\n" + "=" * 80)
print("FEATURE SELECTION RESULTS SUMMARY (Gaussian Naive Bayes)")
print("=" * 80)

# Create results dataframe
results_data = {}
for n_features in feature_counts:
    results_data[n_features] = {
        'Accuracy': f"{feature_selection_results_gnb[n_features]['accuracy'].mean():.3f} ± {feature_selection_results_gnb[n_features]['accuracy'].std():.3f}",
        'Precision': f"{feature_selection_results_gnb[n_features]['precision'].mean():.3f} ± {feature_selection_results_gnb[n_features]['precision'].std():.3f}",
        'Recall': f"{feature_selection_results_gnb[n_features]['recall'].mean():.3f} ± {feature_selection_results_gnb[n_features]['recall'].std():.3f}",
        'F1 Score': f"{feature_selection_results_gnb[n_features]['f1'].mean():.3f} ± {feature_selection_results_gnb[n_features]['f1'].std():.3f}",
        'ROC AUC': f"{feature_selection_results_gnb[n_features]['roc_auc'].mean():.3f} ± {feature_selection_results_gnb[n_features]['roc_auc'].std():.3f}"
    }

results_df = pd.DataFrame(results_data).T
print("\n")
print(results_df)

# Find best feature count based on Accuracy
best_feature_count_gnb = max(
    feature_selection_results_gnb.keys(),
    key=lambda x: feature_selection_results_gnb[x]['accuracy'].mean()
)

best_accuracy = feature_selection_results_gnb[best_feature_count_gnb]['accuracy'].mean()
best_accuracy_std = feature_selection_results_gnb[best_feature_count_gnb]['accuracy'].std()

print("\n" + "=" * 80)
print("BEST FEATURE COUNT")
print("=" * 80)
print(f"\nBest Number of Features: {best_feature_count_gnb}")
print(f"Best Accuracy: {best_accuracy:.3f} ± {best_accuracy_std:.3f}")
print("\nAll Metrics for Best Feature Count:")
print("-" * 80)
for metric_name in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    metric_values = feature_selection_results_gnb[best_feature_count_gnb][metric_name]
    mean = metric_values.mean()
    std = metric_values.std()
    print(f"  {metric_name.upper():10s}: {mean:.3f} ± {std:.3f}")

print("=" * 80)

# Store best feature count for next step
best_features_gnb = best_feature_count_gnb
print(f"\n✓ Best feature count saved: {best_features_gnb}")


Testing with 8 features...
  Accuracy: 0.7212 ± 0.0054

Testing with 9 features...
  Accuracy: 0.7180 ± 0.0052

Testing with 10 features...
  Accuracy: 0.7186 ± 0.0052

Testing with 11 features...
  Accuracy: 0.7188 ± 0.0055

Testing with 12 features...
  Accuracy: 0.7187 ± 0.0053

Testing with 13 features...
  Accuracy: 0.7187 ± 0.0053

Testing with 14 features...
  Accuracy: 0.7190 ± 0.0050

FEATURE SELECTION RESULTS SUMMARY (Gaussian Naive Bayes)


         Accuracy      Precision         Recall       F1 Score        ROC AUC
8   0.721 ± 0.005  0.731 ± 0.005  0.721 ± 0.005  0.718 ± 0.006  0.785 ± 0.006
9   0.718 ± 0.005  0.727 ± 0.005  0.718 ± 0.005  0.714 ± 0.005  0.781 ± 0.005
10  0.719 ± 0.005  0.727 ± 0.005  0.719 ± 0.005  0.715 ± 0.005  0.782 ± 0.005
11  0.719 ± 0.005  0.728 ± 0.005  0.719 ± 0.005  0.715 ± 0.006  0.783 ± 0.005
12  0.719 ± 0.005  0.728 ± 0.005  0.719 ± 0.005  0.715 ± 0.006  0.783 ± 0.005
13  0.719 ± 0.005  0.728 ± 0.005  0.719 ± 0.005  0.715 ± 0.006  0.783 ± 0.0

# Final GNB Pipeline:

In [20]:
# Stratified 5-Fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

# Store metrics for all folds
fold_accuracies = []
fold_precisions = []
fold_recalls = []
fold_f1s = []
fold_roc_aucs = []

print("\nRunning 5-Fold Stratified Cross-Validation...")
print("-" * 80)
print(f"{'Fold':<6} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1 Score':<12} {'ROC AUC':<12}")
print("-" * 80)

# CV Loop - FEATURE SELECTION EMBEDDED INSIDE
fold_idx = 0
for train_idx, val_idx in cv.split(X_train, y_train):
    fold_idx += 1
    
    # Split data into train and validation
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Step 1: Scale the training fold
    scaler = StandardScaler()
    X_fold_train_scaled = scaler.fit_transform(X_fold_train)
    X_fold_val_scaled = scaler.transform(X_fold_val)
    
    # Step 2: Select best features using SelectKBest (fit on training fold only)
    selector = SelectKBest(score_func=f_classif, k=best_features_gnb)
    X_fold_train_selected = selector.fit_transform(X_fold_train_scaled, y_fold_train)
    X_fold_val_selected = selector.transform(X_fold_val_scaled)
    
    # Step 3: Train GNB model on selected features
    model = GaussianNB(var_smoothing=best_params_gnb['var_smoothing'])
    model.fit(X_fold_train_selected, y_fold_train)
    
    # Step 4: Predict on validation fold
    y_pred = model.predict(X_fold_val_selected)
    y_pred_proba = model.predict_proba(X_fold_val_selected)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_fold_val, y_pred)
    precision = precision_score(y_fold_val, y_pred, average='weighted')
    recall = recall_score(y_fold_val, y_pred, average='weighted')
    f1 = f1_score(y_fold_val, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_fold_val, y_pred_proba, average='weighted')
    
    # Store results
    fold_accuracies.append(accuracy)
    fold_precisions.append(precision)
    fold_recalls.append(recall)
    fold_f1s.append(f1)
    fold_roc_aucs.append(roc_auc)
    
    # Print fold results
    print(f"{fold_idx:<6} {accuracy:<12.4f} {precision:<12.4f} {recall:<12.4f} {f1:<12.4f} {roc_auc:<12.4f}")

# Calculate and display summary
print("\n" + "=" * 80)
print("FINAL PIPELINE RESULTS (Mean ± Std)")
print("=" * 80)

print("\n")
print(f"{'Accuracy':<12s}: {np.mean(fold_accuracies):.3f} ± {np.std(fold_accuracies):.3f}")
print(f"{'Precision':<12s}: {np.mean(fold_precisions):.3f} ± {np.std(fold_precisions):.3f}")
print(f"{'Recall':<12s}: {np.mean(fold_recalls):.3f} ± {np.std(fold_recalls):.3f}")
print(f"{'F1 Score':<12s}: {np.mean(fold_f1s):.3f} ± {np.std(fold_f1s):.3f}")
print(f"{'ROC AUC':<12s}: {np.mean(fold_roc_aucs):.3f} ± {np.std(fold_roc_aucs):.3f}")

print("\n" + "=" * 80)

# ============================================================
# TRAIN FINAL MODEL ON FULL TRAIN DATA
# ============================================================

print("Training Final Model on full train data...")

# Step 1: Scale the full training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Step 2: Select best features using SelectKBest on full training data
selector = SelectKBest(score_func=f_classif, k=best_features_gnb)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)

# Step 3: Train final GNB model on selected features
final_model_gnb = GaussianNB(var_smoothing=best_params_gnb['var_smoothing'])
final_model_gnb.fit(X_train_selected, y_train)

print("✓ Final model trained on full training data!")
print("=" * 80)

# Store components for test evaluation
final_scaler_gnb = scaler
final_selector_gnb = selector


Running 5-Fold Stratified Cross-Validation...
--------------------------------------------------------------------------------
Fold   Accuracy     Precision    Recall       F1 Score     ROC AUC     
--------------------------------------------------------------------------------
1      0.7310       0.7402       0.7310       0.7277       0.7961      
2      0.7166       0.7267       0.7166       0.7125       0.7819      
3      0.7223       0.7309       0.7223       0.7190       0.7866      
4      0.7158       0.7245       0.7158       0.7123       0.7786      
5      0.7205       0.7310       0.7205       0.7165       0.7817      

FINAL PIPELINE RESULTS (Mean ± Std)


Accuracy    : 0.721 ± 0.005
Precision   : 0.731 ± 0.005
Recall      : 0.721 ± 0.005
F1 Score    : 0.718 ± 0.006
ROC AUC     : 0.785 ± 0.006

Training Final Model on full train data...
✓ Final model trained on full training data!
